# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal check 1: CTR vs. position (behind the CTR-fix flag)

Hypothesis: pages ranking well (good position) should get good CTR: worse
position -> lower CTR, roughly monotonic. If a page breaks this pattern
(good position, bad CTR), it's a real CTR-fix candidate.

### Signal check 2: Staleness (behind the refresh flag)

Hypothesis: older content (more days since creation, no updates) is more
likely to be declining by month's end.

In [1]:
import duckdb, pandas as pd
con = duckdb.connect()
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
fact_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

# --- Signal check 1: CTR vs position bucket table ---
signal1 = con.sql(f"""
WITH daily AS (
    SELECT * FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
),
first_half AS (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS avg_position,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks
    FROM daily WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 50
)
SELECT
    CASE
        WHEN avg_position <= 3 THEN '1-3'
        WHEN avg_position <= 10 THEN '4-10'
        WHEN avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(clicks * 1.0 / NULLIF(impressions,0)), 4) AS mean_ctr
FROM first_half
GROUP BY 1
ORDER BY 1
""").df()
print("Signal check 1 — CTR by position bucket:")
print(signal1)

# --- Signal check 2: staleness (content age) vs decline rate ---
signal2 = con.sql(f"""
WITH daily AS (
    SELECT * FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
),
first_half AS (
    SELECT client_hash_id, content_hash_id, AVG(gsc_avg_position) AS fh_pos
    FROM daily WHERE report_date <= DATE '2026-03-15' GROUP BY 1,2
),
second_half AS (
    SELECT client_hash_id, content_hash_id, AVG(gsc_avg_position) AS sh_pos
    FROM daily WHERE report_date > DATE '2026-03-15' GROUP BY 1,2
),
joined AS (
    SELECT f.client_hash_id, f.content_hash_id, f.fh_pos, s.sh_pos,
           CASE WHEN s.sh_pos > f.fh_pos THEN 1 ELSE 0 END AS declining,
           DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') AS age_days
    FROM first_half f
    JOIN second_half s USING (client_hash_id, content_hash_id)
    JOIN read_parquet('{rel}/dim_content.parquet') c USING (client_hash_id, content_hash_id)
)
SELECT
    CASE
        WHEN age_days < 90 THEN '0-90'
        WHEN age_days < 180 THEN '91-180'
        WHEN age_days < 365 THEN '181-365'
        ELSE '365+'
    END AS age_bucket,
    COUNT(*) AS n,
    ROUND(AVG(declining), 3) AS decline_rate
FROM joined
WHERE age_days >= 0
GROUP BY 1
ORDER BY 1
""").df()
print("\nSignal check 2 — decline rate by content-age bucket:")
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal check 1 — CTR by position bucket:
  position_bucket      n  mean_ctr
0             1-3  10500    0.0040
1           11-20  18428    0.0027
2             21+  20638    0.0014
3            4-10  42982    0.0034


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Signal check 2 — decline rate by content-age bucket:
  age_bucket      n  decline_rate
0       0-90  43658         0.564
1    181-365  54433         0.532
2       365+  11378         0.586
3     91-180  24809         0.491


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


**The rule:** flag content whose actual CTR falls below the confirmed
position-bucket benchmark CTR, weighted by impression volume (so a big
gap on a high-traffic page ranks above the same gap on a barely-seen
page). Staleness was tested and rejected (MIXED, no clean trend) — this
rule uses only the confirmed signal.

**Score:** estimated missed clicks = (benchmark_ctr - actual_ctr) x
impressions, first half of March only.
**Reason code:** `CTR_BELOW_POSITION_BENCHMARK` (fixed, one code for this
rule).
**Action label:** `review_ctr_fix` (fixed, one action for this rule).

In [2]:
import os

# Bucket benchmark CTR, computed the same way as signal check 1
benchmarks = con.sql(f"""
WITH daily AS (
    SELECT * FROM read_parquet('{fact_path}', hive_partitioning=1)
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
),
first_half AS (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS avg_position,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks
    FROM daily WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 50
)
SELECT *,
    CASE
        WHEN avg_position <= 3 THEN '1-3'
        WHEN avg_position <= 10 THEN '4-10'
        WHEN avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,
    clicks * 1.0 / NULLIF(impressions,0) AS actual_ctr
FROM first_half
""").df()

bucket_benchmark = benchmarks.groupby("position_bucket")["actual_ctr"].mean().rename("benchmark_ctr")
benchmarks = benchmarks.join(bucket_benchmark, on="position_bucket")

benchmarks["reason_code"] = "CTR_BELOW_POSITION_BENCHMARK"
benchmarks["action"] = "review_ctr_fix"
benchmarks["score"] = (benchmarks["benchmark_ctr"] - benchmarks["actual_ctr"]) * benchmarks["impressions"]

queue = benchmarks[benchmarks["score"] > 0].sort_values("score", ascending=False)
queue = queue[["client_hash_id", "content_hash_id", "position_bucket", "avg_position",
               "impressions", "actual_ctr", "benchmark_ctr", "score", "reason_code", "action"]]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Queue rows:", len(queue))
queue.head(10)

Queue rows: 64458


,client_hash_id,content_hash_id,position_bucket,avg_position,impressions,actual_ctr,benchmark_ctr,score,reason_code,action
84218,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,4-10,8.607910,83772.0,0.000000,0.003384,283.504616,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
6661,client_62f4a7e64f5e0096,content_34a70fea29d15f24,1-3,2.786744,73639.0,0.000244,0.004026,278.446414,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
32904,client_62f4a7e64f5e0096,content_7c6373141eae744a,4-10,5.785512,86860.0,0.000587,0.003384,242.955151,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
46536,client_73cda7b4e4f265ea,content_8e1334d6356668e3,4-10,4.579049,58553.0,0.000017,0.003384,197.157449,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
61466,client_23a62021009f63c4,content_65c75874a23fca87,4-10,9.013531,55680.0,0.000269,0.003384,173.434525,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
6681,client_62f4a7e64f5e0096,content_945d6ff91386c817,4-10,6.413782,49314.0,0.000041,0.003384,164.890448,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
70952,client_62f4a7e64f5e0096,content_f6116743b00afc2d,4-10,9.493028,49619.0,0.000161,0.003384,159.922642,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
6672,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,4-10,4.011050,52378.0,0.000344,0.003384,159.259762,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
36709,client_20259bd6705d81d4,content_82e35c4845e6c391,11-20,18.269589,70169.0,0.000413,0.002674,158.663719,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix
32932,client_62f4a7e64f5e0096,content_acbcc847f8996314,4-10,3.453361,83715.0,0.001589,0.003384,150.311714,CTR_BELOW_POSITION_BENCHMARK,review_ctr_fix


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
## 3. Top-10 review

1. **content_9c057b66c30a3abb** (client_73cda7b4e4f265ea) — action:
   review_ctr_fix. Why: position 8.6, 83,772 impressions, but exactly
   0.000000 CTR — the single largest gap in the queue. What would make
   it wrong: exact-zero CTR at this volume is more consistent with a
   tracking/data-pull issue than a real on-page problem — verify GSC
   clicks aren't being miscounted before treating this as fixable.

2. **content_34a70fea29d15f24** (client_62f4a7e64f5e0096) — action:
   review_ctr_fix. Why: position 2.79 (top bucket), 73,639 impressions,
   CTR of 0.000244 — a page ranking this well should not have near-zero
   clicks. What would make it wrong: a title/meta rewrite (the usual
   CTR-fix) won't help if a rich result or brand SERP feature is
   absorbing the clicks instead.

3. **content_7c6373141eae744a** (client_62f4a7e64f5e0096) — action:
   review_ctr_fix. Why: position 5.79, 86,860 impressions, CTR 0.000587.
   What would make it wrong: same client as #2 — two appearances this
   high suggests a client-level tracking issue, not two independent
   content problems.

4. **content_8e1334d6356668e3** (client_73cda7b4e4f265ea) — action:
   review_ctr_fix. Why: position 4.58, 58,553 impressions, CTR 0.000017
   (essentially zero). What would make it wrong: same client as #1 —
   two of the top four rows sharing a client points at a client-wide
   integration issue rather than two separate pages needing fixes.

5. **content_65c75874a23fca87** (client_23a62021009f63c4) — action:
   review_ctr_fix. Why: position 9.01, 55,680 impressions, CTR 0.000269
   — a genuinely different client, so this one reads as a more credible
   individual candidate. What would make it wrong: need to confirm the
   SERP snippet isn't already well-optimized and that no competing
   feature (image pack, ad block) is structurally suppressing clicks.

6. **content_945d6ff91386c817** (client_62f4a7e64f5e0096) — action:
   review_ctr_fix. Why: position 6.41, 49,314 impressions, CTR 0.000041.
   What would make it wrong: third appearance of this same client in six
   rows — strengthens the case that this is a client-level anomaly, not
   six independent content issues.

7. **content_f6116743b00afc2d** (client_62f4a7e64f5e0096) — action:
   review_ctr_fix. Why: position 9.49, 49,619 impressions, CTR 0.000161.
   What would make it wrong: fourth occurrence of the same client —
   at this point the pattern itself, not any individual page, is the
   real finding.

8. **content_1642f339bd6e7c8d** (client_62f4a7e64f5e0096) — action:
   review_ctr_fix. Why: position 4.01, 52,378 impressions, CTR 0.000344.
   What would make it wrong: fifth occurrence of the same client — same
   reasoning as #6 and #7.

9. **content_82e35c4845e6c391** (client_20259bd6705d81d4) — action:
   review_ctr_fix. Why: position 18.27 (bucket 11-20), 70,169
   impressions, CTR 0.000413 vs. a benchmark of only 0.0027. What would
   make it wrong: at position ~18 even the benchmark CTR is already low,
   so the raw-impression-weighted score may be overstating how
   actionable this one really is compared to a near-top-of-page page
   with the same score.

10. **content_acbcc847f8996314** (client_62f4a7e64f5e0096) — action:
    review_ctr_fix. Why: position 3.45, 83,715 impressions, CTR 0.001589
    — not zero like the others, but still well below benchmark. What
    would make it wrong: sixth appearance of this same client in the
    top 10 — by this point, a client-level check should happen before
    treating this as an individual content decision.

In [3]:
# Show the actual top-10 rows the review above is based on
top10 = queue.head(10)[["content_hash_id", "client_hash_id", "position_bucket",
                         "avg_position", "impressions", "actual_ctr",
                         "benchmark_ctr", "score"]]
print(top10.to_string(index=False))

         content_hash_id          client_hash_id position_bucket  avg_position  impressions  actual_ctr  benchmark_ctr      score
content_9c057b66c30a3abb client_73cda7b4e4f265ea            4-10      8.607910      83772.0    0.000000       0.003384 283.504616
content_34a70fea29d15f24 client_62f4a7e64f5e0096             1-3      2.786744      73639.0    0.000244       0.004026 278.446414
content_7c6373141eae744a client_62f4a7e64f5e0096            4-10      5.785512      86860.0    0.000587       0.003384 242.955151
content_8e1334d6356668e3 client_73cda7b4e4f265ea            4-10      4.579049      58553.0    0.000017       0.003384 197.157449
content_65c75874a23fca87 client_23a62021009f63c4            4-10      9.013531      55680.0    0.000269       0.003384 173.434525
content_945d6ff91386c817 client_62f4a7e64f5e0096            4-10      6.413782      49314.0    0.000041       0.003384 164.890448
content_f6116743b00afc2d client_62f4a7e64f5e0096            4-10      9.493028      49619.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
## 4. Weak picks + leakage check

**Weak pattern found:** `client_62f4a7e64f5e0096` accounts for 6 of the
top 10 rows. Combined with several rows showing exactly 0.000000 or
near-zero CTR at 50k-86k impressions and strong positions, this looks
more like a client-level GSC tracking or data-integration problem than
six independent content issues. Before acting on any of this client's
rows individually, I'd check whether their click tracking is broken
site-wide — if so, this rule is currently surfacing a data-quality bug,
not six real CTR-fix opportunities.

**Leakage check:** the rule uses only first-half March data
(`report_date <= 2026-03-15`) for both the score and the benchmark — no
second-half data, no `declining_flag`, and no future window touches the
score. `content_created_date` (used only in Signal Check 2, not in the
final rule) is a static historical fact, not a future-derived field. No
product-computed flags were used as inputs. The rule is independent of
the label-derived checks from Section 1.

In [4]:
# Back up the "weak pattern" claim: how concentrated is one client in the top 10?
client_counts = queue.head(10)["client_hash_id"].value_counts()
print("Client frequency in top 10:")
print(client_counts)

# Back up the leakage check: confirm which columns actually feed the score
print("\nColumns used in the rule/score computation:")
print(list(benchmarks.columns))
print("\nConfirm no second-half or label-derived columns present:",
      not any(col in benchmarks.columns for col in ["sh_pos", "declining_flag", "second_half_avg_position"]))

Client frequency in top 10:
client_hash_id
client_62f4a7e64f5e0096    6
client_73cda7b4e4f265ea    2
client_23a62021009f63c4    1
client_20259bd6705d81d4    1
Name: count, dtype: int64

Columns used in the rule/score computation:
['client_hash_id', 'content_hash_id', 'avg_position', 'impressions', 'clicks', 'position_bucket', 'actual_ctr', 'benchmark_ctr', 'reason_code', 'action', 'score']

Confirm no second-half or label-derived columns present: True


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.